# Vibe Coding Scraper
Minimal pipeline for SpecStory traces and GitHub metadata.

In [1]:
import os
import time
import json
import requests
from urllib.parse import quote

API_BASE_URL = "https://api.github.com"
GITHUB_PAT = os.getenv("GITHUB_PAT", "YOUR_PAT_HERE")

RUN_SEARCHES = False
RUN_REPOSITORIES = False
RUN_LANGUAGES = False
RUN_CONTRIBUTORS = True
RUN_READMES = False
RUN_FILE_TREES = True
RUN_CONTENTS = True
RUN_MARKDOWNS = False
RUN_COMMITS_HISTORY = True
RUN_COMMITS_PATH = True
RUN_COMMITS = True

## Search Collection
Query `.specstory/history/*` markdown paths from GitHub search.

In [2]:
from datetime import datetime, timedelta


def make_request(url, max_retries=5):
    """
    Make a GET request with retry logic.
    If rate limited (403) or transient error, wait and retry.
    """
    headers = {"Authorization": f"Bearer {GITHUB_PAT}"}
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.HTTPError as e:
            status = response.status_code
            if status == 403:  # rate limit or forbidden
                retry_after = response.headers.get("Retry-After")
                if retry_after and retry_after.isdigit():
                    sleep_time = max(int(retry_after), 1)
                else:
                    reset_time = int(
                        response.headers.get("X-RateLimit-Reset", time.time() + 10)
                    )
                    sleep_time = max(reset_time - int(time.time()), 5)
                print(
                    f"⚠️ Rate limit reached (403). Sleeping for {sleep_time} seconds..."
                )
                time.sleep(sleep_time)
                retries += 1
                continue
            else:
                print(f"❌ HTTP error {status}: {e}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Request error: {e}, retrying in 30s...")
            time.sleep(30)
            retries += 1
    print("❌ Max retries reached, giving up on this request.")
    return None


def fetch_prefix(prefix):
    query = f".specstory/history/{prefix} in:path language:markdown"
    encoded_query = quote(query)
    all_items = []
    page = 1

    while True:
        url = f"{API_BASE_URL}/search/code?q={encoded_query}&per_page=100&page={page}"
        data = make_request(url)
        if not data:
            break
        items = data.get("items", [])
        if not items:
            break
        all_items.extend(items)
        print(
            f"Prefix {prefix}: page {page}, got {len(items)} items (total {data.get('total_count', 0)})"
        )

        # stop if last page or API max (1100 cap)
        if len(items) < 100 or len(all_items) >= 1100:
            break
        page += 1
        time.sleep(10)

    os.makedirs("data/searches", exist_ok=True)
    out_path = f"data/searches/{prefix}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(all_items, f, ensure_ascii=False, indent=2)

    print(f"✅ Prefix {prefix}, Got {len(all_items)} saved to {out_path}")


def generate_daily_prefixes(start_date, end_date):
    start_dt = datetime.strptime(start_date, "%Y-%m-%d").date()
    end_dt = datetime.strptime(end_date, "%Y-%m-%d").date()
    if start_dt > end_dt:
        raise ValueError("START_DATE must be earlier than or equal to END_DATE")
    total_days = (end_dt - start_dt).days + 1
    return [
        (start_dt + timedelta(days=offset)).strftime("%Y-%m-%d")
        for offset in range(total_days)
    ]


START_DATE = "2025-01-01"
END_DATE = "2026-04-30"
PREFIXES = generate_daily_prefixes(START_DATE, END_DATE)
# PREFIXES = ["2024-09", "2024-10", "2024-11", "2024-12"]

if RUN_SEARCHES:
    # print(f"Running searches from {START_DATE} to {END_DATE} ({len(PREFIXES)} days)")
    for prefix in PREFIXES:
        fetch_prefix(prefix)
else:
    print("Searches not run. Use existing search results.")

Searches not run. Use existing search results.


## Search Aggregation
Merge prefix files and build a unique file index.

In [3]:
def load_all_items(search_dir="data/searches"):
    all_items = []
    monthly_counts = {}

    for filename in sorted(os.listdir(search_dir)):
        if filename.endswith(".json"):
            with open(os.path.join(search_dir, filename), "r") as f:
                data = json.load(f)
                all_items.extend(data)

            month_key = os.path.splitext(filename)[0][:7]
            monthly_counts[month_key] = monthly_counts.get(month_key, 0) + len(data)

    return all_items, monthly_counts


all_items, monthly_counts = load_all_items()
print(f"Total items loaded: {len(all_items)}")
print("Monthly totals:")
for month in sorted(monthly_counts):
    print(f"{month}: {monthly_counts[month]}")

with open("data/searches.json", "w") as f:
    json.dump(all_items, f, indent=2)

Total items loaded: 17592
Monthly totals:
2024-09: 5
2024-10: 23
2024-11: 14
2024-12: 96
2025-01: 78
2025-02: 365
2025-03: 827
2025-04: 931
2025-05: 1380
2025-06: 4570
2025-07: 1438
2025-08: 927
2025-09: 1195
2025-10: 1035
2025-11: 816
2025-12: 953
2026-01: 688
2026-02: 787
2026-03: 804
2026-04: 660


In [4]:
import pandas as pd

all_items = json.load(open("data/searches.json"))
rows = []
for item in all_items:
    rows.append(
        {
            "name": item.get("name"),
            "path": item.get("path"),
            "sha": item.get("sha"),
            "url": item.get("url"),
            "git_url": item.get("git_url"),
            "repository_full_name": item["repository"]["full_name"],
            "repository_url": item["repository"]["url"],
        }
    )

df = pd.DataFrame(rows)
df = df.sort_values(by="name").reset_index(drop=True)
df = df.drop_duplicates(subset=["sha"]).reset_index(drop=True)
print(f"Total unique markdown files: {len(df)}")

Total unique markdown files: 14789


## Repository Metadata
Fetch repositories and normalize core repository fields.

In [5]:
from tqdm import tqdm

repository_counts = df["repository_url"].value_counts().sort_values(ascending=False)
repo_urls = repository_counts.index.tolist()

if RUN_REPOSITORIES:
    print(f"Fetching {len(repo_urls)} repositories...")
    repo_infos = []
    for repo_url in tqdm(repo_urls):
        try:
            r = requests.get(
                repo_url, headers={"Authorization": f"Bearer {GITHUB_PAT}"}
            )
            r.raise_for_status()
            data = r.json()
            repo_infos.append(data)
        except Exception as e:
            print(f"Error fetching {repo_url}: {e}, skipping.")

    with open("data/repositories.json", "w") as f:
        json.dump(repo_infos, f, indent=2)
else:
    print("Repositories not fetched. Use existing repositories.json.")

Repositories not fetched. Use existing repositories.json.


In [6]:
import pandas as pd

all_items = json.load(open("data/repositories.json"))
rows = []
for item in all_items:
    license_info = item.get("license") or {}
    rows.append(
        {
            "id": item.get("id"),
            "name": item.get("name"),
            "full_name": item.get("full_name"),
            "languages_url": item.get("languages_url"),
            "commits_url": item.get("commits_url").replace("{/sha}", ""),
        }
    )

df_repositories = pd.DataFrame(rows)
print(f"Total repositories: {len(df_repositories)}")

Total repositories: 1441


## Session-to-Repository Mapping
Build an accurate mapping between each session and its repository into a single table.

In [7]:
# Reuse the same deduplicated session table from the search aggregation step.
df_mapping = df[["name", "sha", "repository_full_name"]].rename(
    columns={"name": "session_name", "sha": "session_sha"}
)

# Merge session rows with repository ids for downstream analysis.
df_mapping = df_mapping.merge(
    df_repositories[["id", "full_name"]].rename(columns={"id": "repository_id"}),
    left_on="repository_full_name",
    right_on="full_name",
    how="left",
)
df_mapping = df_mapping.drop(
    columns=["full_name", "session_name", "repository_full_name"]
)
df_mapping = df_mapping.sort_values(by=["repository_id"]).reset_index(drop=True)
df_mapping.to_csv("data/mapping.csv", index=False)
print(f"Total session-to-repository mappings: {len(df_mapping)}")

Total session-to-repository mappings: 14789


In [8]:
if RUN_LANGUAGES:
    print("Fetching languages for repositories...")
    language_data = []
    for row in tqdm(df_repositories.itertuples(), total=len(df_repositories)):
        lang_url = row.languages_url
        try:
            r = requests.get(
                lang_url, headers={"Authorization": f"Bearer {GITHUB_PAT}"}
            )
            r.raise_for_status()
            data = r.json()
            language_data.append({"full_name": row.full_name, "languages": data})
        except Exception as e:
            print(f"Error fetching languages for {row.full_name}: {e}, skipping.")

    with open("data/languages.json", "w") as f:
        json.dump(language_data, f, indent=2)
else:
    language_data = json.load(open("data/languages.json"))
    print("Languages not fetched. Use existing languages.json.")
print(f"Total repositories with language data: {len(language_data)}")

Languages not fetched. Use existing languages.json.
Total repositories with language data: 1441


## Repo Enrichment
Collect languages, contributors, READMEs, and file trees.

In [9]:
if RUN_CONTRIBUTORS:
    os.makedirs("data/contributors", exist_ok=True)
    print(f"Fetching contributors for {len(df_repositories)} repositories...")

    existing_contributors = {
        f[:-5] for f in os.listdir("data/contributors") if f.endswith(".json")
    }
    df_to_fetch = df_repositories[
        ~df_repositories["id"].astype(str).isin(existing_contributors)
    ]
    print(f"{len(df_to_fetch)} repositories to fetch after excluding existing files.")

    for row in tqdm(df_to_fetch.itertuples(), total=len(df_to_fetch)):
        all_contributors = []
        page = 1
        while True:
            contributors_url = f"{API_BASE_URL}/repos/{row.full_name}/contributors?per_page=100&page={page}"
            data = make_request(contributors_url)
            if data is None:
                break
            if not data:
                break
            all_contributors.extend(data)
            if len(data) < 100:
                break
            page += 1

        out_path = f"data/contributors/{row.id}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(all_contributors, f, ensure_ascii=False, indent=2)
else:
    print("Contributors not fetched. Use existing contributors in data/contributors/.")

Fetching contributors for 1441 repositories...
0 repositories to fetch after excluding existing files.


0it [00:00, ?it/s]


In [10]:
import base64
from tqdm import tqdm

if RUN_READMES:
    os.makedirs("data/readmes", exist_ok=True)
    print(f"Fetching READMEs for {len(df_repositories)} repositories...")

    existing_readmes = {f[:-3] for f in os.listdir("data/readmes") if f.endswith(".md")}
    df_to_fetch = df_repositories[
        ~df_repositories["id"].astype(str).isin(existing_readmes)
    ]
    print(f"{len(df_to_fetch)} repositories to fetch after excluding existing files.")

    for row in tqdm(df_to_fetch.itertuples(), total=len(df_to_fetch)):
        repo_id = row.id
        repo_full_name = row.full_name
        readme_url = f"{API_BASE_URL}/repos/{repo_full_name}/readme"

        try:
            r = requests.get(
                readme_url, headers={"Authorization": f"Bearer {GITHUB_PAT}"}
            )
            r.raise_for_status()
            data = r.json()

            encoded_content = data.get("content", "")
            decoded_content = ""
            if encoded_content:
                try:
                    decoded_content = base64.b64decode(encoded_content).decode(
                        "utf-8", errors="ignore"
                    )
                except Exception as decode_error:
                    print(
                        f"Error decoding README content for {repo_full_name} ({repo_id}): {decode_error}"
                    )

            with open(f"data/readmes/{repo_id}.md", "w", encoding="utf-8") as f:
                f.write(decoded_content)

        except Exception as e:
            continue
else:
    print("READMEs not fetched. Use existing files in data/readmes/.")

READMEs not fetched. Use existing files in data/readmes/.


In [11]:
from tqdm import tqdm


def get_file_tree(repo_full_name, token=None):
    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"

    # Get the default branch first.
    repo_url = f"https://api.github.com/repos/{repo_full_name}"
    repo_resp = requests.get(repo_url, headers=headers)
    if repo_resp.status_code != 200:
        return None

    branch = repo_resp.json().get("default_branch")
    if not branch:
        return None

    # Get the full tree recursively.
    tree_url = (
        f"https://api.github.com/repos/{repo_full_name}/git/trees/{branch}?recursive=1"
    )
    tree_resp = requests.get(tree_url, headers=headers)
    if tree_resp.status_code != 200:
        return None

    data = tree_resp.json()
    paths = [item["path"] for item in data.get("tree", []) if "path" in item]
    return paths


if RUN_FILE_TREES:
    os.makedirs("data/file_trees", exist_ok=True)
    print(f"Fetching file trees for {len(df_repositories)} repositories...")

    existing_trees = {
        f[:-5] for f in os.listdir("data/file_trees") if f.endswith(".json")
    }
    df_to_fetch = df_repositories[
        ~df_repositories["id"].astype(str).isin(existing_trees)
    ]
    print(f"{len(df_to_fetch)} repositories to fetch after excluding existing files.")

    token = None if GITHUB_PAT == "YOUR_PAT_HERE" else GITHUB_PAT

    for row in tqdm(df_to_fetch.itertuples(), total=len(df_to_fetch)):
        repo_id = row.id
        repo_full_name = row.full_name
        paths = get_file_tree(repo_full_name, token=token)

        if paths is None:
            continue

        with open(f"data/file_trees/{repo_id}.json", "w", encoding="utf-8") as f:
            json.dump(paths, f, ensure_ascii=False, indent=2)
else:
    print("File trees not fetched. Use existing files in data/file_trees/.")

Fetching file trees for 1441 repositories...
0 repositories to fetch after excluding existing files.


0it [00:00, ?it/s]


## Chat Content
Fetch chat blobs and optionally decode markdown files.

In [12]:
from tqdm import tqdm

if RUN_CONTENTS:
    os.makedirs("data/contents", exist_ok=True)
    print(f"Fetching contents for {len(df)} files...")
    existing_files = set(os.listdir("data/contents"))
    df_to_fetch = df[~df["sha"].isin(existing_files)].reset_index(drop=True)
    print(f"Fetching {len(df_to_fetch)} new files...")
    for git_url in tqdm(df_to_fetch["git_url"].to_list()):
        try:
            r = requests.get(git_url, headers={"Authorization": f"Bearer {GITHUB_PAT}"})
            r.raise_for_status()
            data = r.json()
            sha = data["sha"]
            with open("data/contents/" + sha, "w") as f:
                f.write(data["content"])
        except Exception as e:
            print(f"Error fetching {git_url}: {e}, skipping.")
            continue

else:
    print("Contents not fetched. Use existing contents in data/contents/.")

Fetching contents for 14789 files...


Fetching 0 new files...


0it [00:00, ?it/s]


In [13]:
import base64

if RUN_MARKDOWNS:
    os.makedirs("data/markdowns", exist_ok=True)
    print("Decoding contents to markdown files...")
    for filename in os.listdir("data/contents"):
        with open(os.path.join("data/contents", filename), "r") as f:
            encoded_content = f.read()
        decoded_content = base64.b64decode(encoded_content).decode(
            "utf-8", errors="ignore"
        )
        with open(
            os.path.join("data/markdowns", filename + ".md"), "w", encoding="utf-8"
        ) as f:
            f.write(decoded_content)
else:
    print("Markdowns not generated. Use existing markdowns in data/markdowns/.")

Markdowns not generated. Use existing markdowns in data/markdowns/.


## Commit Traces
Collect commit history, file-path commits, and commit details.

In [14]:
if RUN_COMMITS_HISTORY:
    os.makedirs("data/commits_history", exist_ok=True)
    print(f"Fetching commit histories for {len(df_repositories)} repositories...")
    existing_repos = [
        f[:-5] for f in os.listdir("data/commits_history") if f.endswith(".json")
    ]
    df_to_fetch = df_repositories[
        ~df_repositories["id"].astype(str).isin(existing_repos)
    ]
    print(f"{len(df_to_fetch)} repositories to fetch after excluding existing files.")

    for row in tqdm(df_to_fetch.itertuples(), total=len(df_to_fetch)):
        commits_url = row.commits_url
        all_commits = []
        page = 1
        while True:
            url = f"{commits_url}?per_page=100&page={page}"
            data = make_request(url)
            if not data:
                break
            all_commits.extend(data)
            if len(data) < 100:
                break
            page += 1

        out_path = f"data/commits_history/{row.id}.json"
        with open(out_path, "w") as f:
            json.dump(all_commits, f, indent=2)
        if page > 1:
            print(
                f"Fetched {len(all_commits)} commits for {row.full_name} in {page} pages."
            )
else:
    print("Commit histories not fetched. Use existing commits_history/*.json.")

Fetching commit histories for 1441 repositories...
0 repositories to fetch after excluding existing files.


0it [00:00, ?it/s]


In [15]:
if RUN_COMMITS_PATH:
    os.makedirs("data/commits_path", exist_ok=True)
    print(f"Fetching commits for {len(df)} files in permissive repositories...")
    existing_commits_path = set(os.listdir("data/commits_path"))
    df_to_fetch = df[~df["sha"].isin(existing_commits_path)].reset_index(drop=True)
    print(f"Fetching {len(df_to_fetch)} new commit paths...")
    for idx, row in tqdm(df_to_fetch.iterrows(), total=len(df_to_fetch)):
        repo_full_name = row["repository_full_name"]
        file_path = row["path"]
        commits_url = f"{API_BASE_URL}/repos/{repo_full_name}/commits?path={quote(file_path)}&per_page=100"
        try:
            r = requests.get(
                commits_url, headers={"Authorization": f"Bearer {GITHUB_PAT}"}
            )
            r.raise_for_status()
            data = r.json()
            sha = row["sha"]
            with open("data/commits_path/" + sha, "w") as f:
                json.dump(data, f, indent=2)
        except Exception as e:
            print(
                f"Error fetching commits for {repo_full_name} {file_path}: {e}, skipping."
            )
            continue
else:
    print("Commits path not fetched. Use existing commits in data/commits_path/.")

Fetching commits for 14789 files in permissive repositories...
Fetching 0 new commit paths...


0it [00:00, ?it/s]


In [16]:
commits_history = []
for sha in os.listdir("data/commits_path"):
    with open(os.path.join("data/commits_path", sha), "r") as f:
        data = json.load(f)
    for commit in data:
        commit_info = {
            "sha": sha,
            "commit_sha": commit.get("sha"),
            "commit_message": commit.get("commit", {}).get("message"),
            "commit_date": commit.get("commit", {}).get("author", {}).get("date"),
            "url": commit.get("url"),
            "parents": commit.get("parents"),
        }
        commits_history.append(commit_info)
df_commits_history = pd.DataFrame(commits_history)
unique_commits = df_commits_history.drop_duplicates(subset=["commit_sha"]).reset_index(
    drop=True
)

if RUN_COMMITS:
    os.makedirs("data/commits", exist_ok=True)
    print(f"Fetching {len(unique_commits)} unique commits...")
    existing_commits = set(os.listdir("data/commits"))
    df_to_fetch = unique_commits[
        ~unique_commits["commit_sha"].isin(existing_commits)
    ].reset_index(drop=True)
    print(f"Fetching {len(df_to_fetch)} new commits...")
    for commit in tqdm(df_to_fetch.itertuples(), total=len(df_to_fetch)):
        commit_url = commit.url
        try:
            r = requests.get(
                commit_url, headers={"Authorization": f"Bearer {GITHUB_PAT}"}
            )
            r.raise_for_status()
            data = r.json()
            sha = commit.commit_sha
            with open("data/commits/" + sha, "w") as f:
                json.dump(data, f, indent=2)
        except Exception as e:
            print(f"Error fetching commit {commit_url}: {e}, skipping.")
            continue
else:
    print("Commits not fetched. Use existing commits in data/commits/.")

Fetching 7206 unique commits...
Fetching 0 new commits...


0it [00:00, ?it/s]
